In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt
import statsmodels.api as sm
import itertools 

In [ ]:
# Imports from China to EU
trade_data = pd.read_csv('data/trade_data_2018_2025.csv')
# Imports from World to EU (incl. China)
trade_data_world = pd.read_csv('data/trade_data_2018_2025_world.csv')

In [ ]:
# Note: Trade data is available only till 2025-09
trade_data.tail()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
1297,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,9.794778e+07,False,0.0,False,7.316084e+08,NaN,7.316084e+08,2,False,True
1298,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,1.202756e+08,True,0.0,False,5.002984e+08,NaN,5.002984e+08,6,False,True
1299,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,2.192306e+08,False,0.0,False,5.343282e+08,NaN,5.343282e+08,2,False,True
1300,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,7.686484e+07,True,0.0,False,8.195563e+08,NaN,8.195563e+08,6,False,True
1301,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,8.777624e+07,True,0.0,False,8.313436e+08,NaN,8.313436e+08,6,False,True


In [47]:
grouped_yr_code = trade_data.groupby(["refYear", "cmdCode", "post_2025"]).agg(
    {"netWgt": "sum"}
)

In [34]:
trade_data['post_2025'] = 0
trade_data['post_2025'] = np.where(trade_data['refYear'] == 2025, 1, 0)

In [39]:
mean_pre_2025 = trade_data[trade_data['post_2025'] == 0]['netWgt'].mean()
mean_post_2025 = trade_data[trade_data['post_2025'] == 1]['netWgt'].mean()

In [54]:
trade_data["normalized_wt"] = np.where(
    trade_data["post_2025"] == 0,
    trade_data["netWgt"] / mean_pre_2025,
    trade_data["netWgt"] / mean_post_2025,
)

In [ ]:
# Std ~0.78 means most values cluster around 0.4–1.6 (25th–75th percentiles).
# Max=4.24 is an outlier (4× mean), but not extreme like raw trade data (where max can be 1000× min).
# No zeros = no need for +1 hack.
# However, max/min ratio is still ~30 so right-skew persists
trade_data['normalized_wt'].describe() 

count    1302.000000
mean        1.000000
std         0.783528
min         0.138602
25%         0.413174
50%         0.795835
75%         1.260649
max         4.239638
Name: normalized_wt, dtype: float64

In [70]:
# β without normalization is "% change in imports/netweight due to the shock."
# Better to think in growth rates ("15% extra xxx demand? not absolute tons ("+50,000 tons?
# With levels, β is in raw kg ("+X tons per 10pp Asia exposure")—hard to compare across products (washing machines vs. toys have different scales).
# Logging makes β comparable and intuitive:
#  "10pp more Asia exposure → ~2% drop in imports." 
trade_data['log_netwgt'] =  np.log(trade_data['netWgt'])

In [78]:
trade_data_world_need = trade_data_world[
    ["refPeriodId", "partnerCode", "partnerDesc", "cmdCode", "netWgt"]
]

In [82]:
trade_data_merged = trade_data.merge(
    trade_data_world_need,
    left_on=["refPeriodId", "cmdCode"],
    right_on=["refPeriodId", "cmdCode"],
    suffixes=(None, "_world"),
)

In [85]:
trade_data_merged["china_share_prod"] = (
    trade_data_merged["netWgt"]/ trade_data_merged["netWgt_world"]
)

In [86]:
trade_data_merged.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,legacyEstimationFlag,isReported,isAggregate,post_2025,normalized_wt,log_netwgt,partnerCode_world,partnerDesc_world,netWgt_world,china_share_prod
0,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,6,False,True,0,0.556667,17.222390,0,World,1.076809e+08,0.280189
1,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,0,False,True,0,0.446041,17.000832,0,World,3.800298e+07,0.636136
2,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,0,False,True,0,1.452619,18.181546,0,World,1.130161e+08,0.696634
3,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,6,False,True,0,0.667876,17.404525,0,World,6.193729e+07,0.584436
4,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,2,False,True,0,0.258116,16.453833,0,World,2.632927e+07,0.531337


In [ ]:
# FIXME
model_log = sm.OLS(
    "log_netwgt ~ interaction + controls + C(product) + C(time)", data=trade_data
).fit()

print(model_levels.summary())
print(model_log.summary())

ValueError: unrecognized data structures: <class 'str'> / <class 'NoneType'>